In [2]:
import json
from collections import defaultdict
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from mlx_lm import load, generate
from tqdm.notebook import tqdm

In [3]:
model, tokenizer = load("RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-mxfp4")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [7]:
SYSTEM_PROMPT_1 = """You are a clinical evidence alignment system.

Your task is to align each answer sentence to supporting clinical note sentences.

Important rules:

1. Alignment is at the answer sentence level.
2. Only include note sentences that DIRECTLY support the answer sentence.
3. Do NOT over-cite.
4. Do NOT under-cite.
5. Do NOT infer beyond the note text.
6. Use only the provided numbered note sentences.
7. If no supporting evidence exists, return an empty list [].
8. Evidence IDs must be strings.
9. Output must be STRICT JSON.
10. Do not include explanations or reasoning.

Output format:

[
  {
    "case_id": "<case_id>",
    "prediction": [
      {
        "answer_id": "<ID>",
        "evidence_id": ["<note_id>", ...]
      }
    ]
  }
]"""

FEWSHOT = """Example:

Case ID: 1

Patient Question:
I had severe abdomen pain and was hospitalised for 15 days in ICU, diagnoised with CBD sludge. Doctor advised for ERCP. My question is if the sludge was there does not any medication help in flushing it out? Whether ERCP was the only cure?

Clinician-Interpreted Question:
Why was ERCP recommended over a medication-based treatment for CBD sludge?

Clinical Note Excerpt:
1: During the ERCP a pancreatic stent was required to facilitate access to the biliary system (removed at the end of the procedure), and a common bile duct stent was placed to allow drainage of the biliary obstruction caused by stones and sludge.
2: However, due to the patient’s elevated INR, no sphincterotomy or stone removal was performed.
3: Frank pus was noted to be draining from the common bile duct, and post-ERCP it was recommended that the patient remain on IV Zosyn for at least a week.
4: The Vancomycin was discontinued.
5: On hospital day 4 (post-procedure day 3) the patient returned to ERCP for re-evaluation of her biliary stent as her LFTs and bilirubin continued an upward trend.
6: On ERCP the previous biliary stent was noted to be acutely obstructed by biliary sludge and stones.
7: As the patient’s INR was normalized to 1.2, a sphincterotomy was safely performed, with removal of several biliary stones in addition to the common bile duct stent.
8: At the conclusion of the procedure, retrograde cholangiogram was negative for filling defects.

Answer:
1: An endoscopic retrograde cholangiopancreatography, ERCP, was recommended to place a common bile duct stent.
2: This stent was placed to allow drainage of the biliary obstruction which was caused by stones and sludge.
3: Due to no improvement in liver function, the patient needed a repeat ERCP.
4: The repeat ERCP showed that the biliary stent placed in the first ERCP was obstructed by stones and sludge.
5: The stones and stent were successfully removed during this procedure by performing a sphincterotomy.

Output:
[
  {
    "case_id": "1",
    "prediction": [
      {
        "answer_id": "1",
        "evidence_id": ["1"]
      },
      {
        "answer_id": "2",
        "evidence_id": ["1"]
      },
      {
        "answer_id": "3",
        "evidence_id": ["5"]
      },
      {
        "answer_id": "4",
        "evidence_id": ["6"]
      },
      {
        "answer_id": "5",
        "evidence_id": ["7"]
      }
    ]
  }
]"""

SYSTEM_PROMPT_1 = SYSTEM_PROMPT_1 + '\n' + FEWSHOT

USER_PROMPT_1 = """Case ID: {}

Patient Question:
{}

Clinician-Interpreted Question:
{}

Clinical Note Excerpt:
{}

Answer:
{}

Provide answer-to-evidence alignment."""

from mlx_lm.sample_utils import make_sampler

sampler = make_sampler(temp=0.2)

def stage_1(*args):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_1},
        {"role": "user", "content": USER_PROMPT_1.format(*args)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False
    )
    
    text = generate(model, tokenizer, prompt=prompt, max_tokens=8192, sampler=sampler)

    return text

In [2]:
with open("./v1.5/dev/archehr-qa_key.json") as f:
    key = json.load(f)


tree = ET.parse("./v1.5/dev/archehr-qa.xml")
root = tree.getroot()

data = []
for case in root.findall("case"):
    case_id = case.get('id')
    c_question = case.findtext("clinician_question", default="").strip()
    p_question = case.findtext("patient_narrative", default="").strip()
    sentences = [
        {
            'sentence_id': sentence.get('id'),
            'text': sentence.text.strip()
        }
        for sentence in case.find("note_excerpt_sentences")
    ]
    item = {
        'case_id': case_id,
        'excerpt_sentences': sentences,
        'clinician_question': c_question,
        'patient_question': p_question,
    }
    data.append(item)

df1 = pd.DataFrame(data)
df2 = pd.DataFrame(key)
combined = df1.merge(df2[['case_id', 'clinician_answer_sentences']], on='case_id').to_dict(orient='records')

In [10]:
from tqdm.notebook import tqdm
from ast import literal_eval

dev_answers = []
for case in tqdm(combined):
    cid = case['case_id']
    pq = case['patient_question']
    cq = case['clinician_question']
    evidence_sentences = '\n'.join(f"{q['sentence_id']}: {q['text'].replace('\n', ' ')}" for q in case['excerpt_sentences'])
    answer_sentences = '\n'.join(f"{q['id']}: {q['text']}" for q in case['clinician_answer_sentences'])
    
    response = stage_1(cid, pq, cq, evidence_sentences, answer_sentences)
    try:
        item = literal_eval(response.split('</think>')[-1])
    except:
        item = [{'case_id': cid, 'prediction': []}]
        print(f"Problem with {cid}")
    
    dev_answers.extend(item)

  0%|          | 0/20 [00:00<?, ?it/s]

In [11]:
from scoring_subtask_4 import compute_alignment_scores, get_leaderboard, load_key

key_map = load_key("./v1.5/dev/archehr-qa_key.json")

scores = compute_alignment_scores(dev_answers, key_map)
leaderboard = get_leaderboard(scores)
leaderboard

{'micro_precision': 31.57894736842105,
 'micro_recall': 21.71945701357466,
 'micro_f1': 25.737265415549597,
 'macro_precision': np.float64(42.60077996715928),
 'macro_recall': np.float64(30.426949347185918),
 'macro_f1': np.float64(34.70617473328611),
 'overall_score': 25.737265415549597}

# Submission Test

In [5]:
with open("./v1.5/test/archehr-qa_key.json") as f:
    key = json.load(f)


tree = ET.parse("./v1.5/test/archehr-qa.xml")
root = tree.getroot()

data = []
for case in root.findall("case"):
    case_id = case.get('id')
    c_question = case.findtext("clinician_question", default="").strip()
    p_question = case.findtext("patient_narrative", default="").strip()
    sentences = [
        {
            'sentence_id': sentence.get('id'),
            'text': sentence.text.strip()
        }
        for sentence in case.find("note_excerpt_sentences")
    ]
    item = {
        'case_id': case_id,
        'excerpt_sentences': sentences,
        'clinician_question': c_question,
        'patient_question': p_question,
    }
    data.append(item)

df1 = pd.DataFrame(data)
df2 = pd.DataFrame(key)
combined = df1.merge(df2[['case_id', 'clinician_answer_sentences']], on='case_id').to_dict(orient='records')

In [8]:
from tqdm.notebook import tqdm
from ast import literal_eval

submission = []
for case in tqdm(combined):
    cid = case['case_id']
    pq = case['patient_question']
    cq = case['clinician_question']
    evidence_sentences = '\n'.join(f"{q['sentence_id']}: {q['text'].replace('\n', ' ')}" for q in case['excerpt_sentences'])
    answer_sentences = '\n'.join(f"{q['id']}: {q['text']}" for q in case['clinician_answer_sentences'])

    response = stage_1(cid, pq, cq, evidence_sentences, answer_sentences)
    try:
        item = literal_eval(response.split('</think>')[-1])
    except:
        item = [{'case_id': cid, 'prediction': []}]
        print(f"Problem with {cid}")

    submission.extend(item)

  0%|          | 0/100 [00:00<?, ?it/s]

In [10]:
with open("./qwen_35_submission/submission_test.json", 'w') as f:
    json.dump(submission, f)

# Submission Test 2026

In [12]:
with open("./v1.5/test-2026/archehr-qa_key.json") as f:
    key = json.load(f)


tree = ET.parse("./v1.5/test-2026/archehr-qa.xml")
root = tree.getroot()

data = []
for case in root.findall("case"):
    case_id = case.get('id')
    c_question = case.findtext("clinician_question", default="").strip()
    p_question = case.findtext("patient_narrative", default="").strip()
    sentences = [
        {
            'sentence_id': sentence.get('id'),
            'text': sentence.text.strip()
        }
        for sentence in case.find("note_excerpt_sentences")
    ]
    item = {
        'case_id': case_id,
        'excerpt_sentences': sentences,
        'clinician_question': c_question,
        'patient_question': p_question,
    }
    data.append(item)

df1 = pd.DataFrame(data)
df2 = pd.DataFrame(key)
combined = df1.merge(df2[['case_id', 'clinician_answer_sentences']], on='case_id').to_dict(orient='records')

In [13]:
from tqdm.notebook import tqdm
from ast import literal_eval

submission = []
for case in tqdm(combined):
    cid = case['case_id']
    pq = case['patient_question']
    cq = case['clinician_question']
    evidence_sentences = '\n'.join(f"{q['sentence_id']}: {q['text'].replace('\n', ' ')}" for q in case['excerpt_sentences'])
    answer_sentences = '\n'.join(f"{q['id']}: {q['text']}" for q in case['clinician_answer_sentences'])
    
    response = stage_1(cid, pq, cq, evidence_sentences, answer_sentences)
    try:
        item = literal_eval(response.split('</think>')[-1])
    except:
        item = [{'case_id': cid, 'prediction': []}]
        print(f"Problem with {cid}")
    
    submission.extend(item)

  0%|          | 0/47 [00:00<?, ?it/s]

In [14]:
with open("./qwen_35_submission/submission2026.json", 'w') as f:
    json.dump(submission, f)

In [15]:
with open("./qwen_35_submission/submission_test.json") as f:
    s1 = json.load(f)
with open("./qwen_35_submission/submission2026.json") as f:
    s2 = json.load(f)
with open("./qwen_35_submission/submission.json", 'w') as f:
    json.dump(s1 + s2, f)